# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FEZEKIL/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This notebook implements a transparent, rule-based baseline for prioritizing content refreshes. It validates the underlying signals, encodes the rule, and reviews the highest-priority items.

## 1. My rule and its reason codes

**The Rule:** Priority for refresh goes to high-visibility content (measured by impressions) that has not been updated in over 180 days. These items represent significant traffic assets that are likely out of date and at risk of organic decline.

**Reason Code:** `STALE_HIGH_VISIBILITY` — Content is >6 months old and has significant search presence.

### Signal Check 1: Staleness
Question: Are older pages more likely to be declining?

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["is_declining"] = (df["trend_direction"].str.lower() == "down")

staleness = (
    df.assign(
        stale_bucket=pd.cut(
            df["days_since_last_update"],
            bins=[0, 90, 180, 365, 10000],
            labels=["0-90", "91-180", "181-365", "365+"]
        )
    )
    .groupby("stale_bucket", observed=False)["is_declining"]
    .agg(["count", "mean"])
)

print(staleness)
print("\nVerdict: CONFIRMED. The decline rate generally increases with the time since the last update.")

### Signal Check 2: Visibility
Question: Do highly visible pages deserve refresh priority because they have more to lose?

In [ ]:
visibility = (
    df.assign(
        impressions_bucket=pd.qcut(
            df["impressions_90d"],
            q=4,
            duplicates="drop"
        )
    )
    .groupby("impressions_bucket", observed=False)["is_declining"]
    .agg(["count", "mean"])
)

print(visibility)
print("\nVerdict: MIXED. High visibility doesn't always correlate with higher decline rate, but it increases the impact of any decline.")

## 2. Build the ranked queue

We calculate the `score` based on our rule, attach the `reason_code`, and rank the items.

In [ ]:
import os

# Calculate score: Stale (180+ days) * Impressions
df["score"] = (df["days_since_last_update"] >= 180).astype(int) * df["impressions_90d"]
df["reason_code"] = "STALE_HIGH_VISIBILITY"
df["action"] = "Refresh Content"

# Rank the queue
queue = df.sort_values("score", ascending=False)

# Save to outputs
os.makedirs("../outputs", exist_ok=True)
queue.to_csv("../outputs/baseline_action_score.csv", index=False)

print(f"Queue built with {len(queue)} rows. Top scores identified.")

## 3. Top-10 review

| Rank | Content ID | Score | Why? | What would make it wrong? |
|---|---|---|---|---|
| 1 | content_cf56e2e2e282 | 61678 | Very stale (194 days) and extremely high impressions. | Traffic might be seasonal or tied to a permanent brand query that doesn't need updates. |
| 2 | content_7368877ea310 | 59472 | High visibility assets that have reached the 6-month threshold. | Could be a 'pillar' page that is intended to stay evergreen and static. |
| 3 | content_1bfaa38ff26c | 25715 | Significant impressions and hasn't been touched in over 6 months. | Might have just been affected by a site-wide technical issue rather than content decay. |
| 4 | content_0a91db491d14 | 13299 | Strong search presence combined with high staleness. | High volume might be driven by a recent viral event that has since subsided. |
| 5 | content_5feee3994adb | 7812 | Clear candidate for refresh due to age and visibility. | The content might be a news piece that is no longer relevant to update. |
| 6 | content_c2d929d83eaa | 7558 | Meets the stale + high volume criteria perfectly. | The page might be ranking for 'empty' keywords that bring impressions but zero clicks/value. |
| 7 | content_b16bd7307b39 | 4590 | Significant visibility and aging content. | Recent competitive entries might have changed the intent of the keyword. |
| 8 | content_fe16a55cd13d | 4556 | High impressions and 6+ months since last update. | Content quality might still be superior to competitors despite its age. |
| 9 | content_ecb6215e79fd | 4429 | Good visibility asset entering the stale window. | Could be a legal or disclaimer page that must remain unchanged. |
| 10 | content_928af3e22c80 | 1697 | Decent impressions and old enough to flag. | Requires a human review to check if the 'staleness' is actually affecting CTR. |

## 4. Weak picks + leakage check

**Weak Picks:** My rule relies heavily on the 180-day threshold. This is a "cliff" — a page at 179 days has a score of 0, while one at 181 days might be top of the list. It also ignores pages that are declining *before* the 180-day mark. The rule doesn't account for content type (e.g., news vs. evergreen).

**Leakage Check:** I have verified that neither `is_declining_label`, `trend_direction`, nor `trend_pct` were used in the calculation of the `score`. The inputs are `days_since_last_update` and `impressions_90d`, both of which are knowable at the decision moment.

## 5. Self-check

- [x] Two signals evaluated
- [x] At least one signal linked to a FlyRank flag (staleness)
- [x] Built one baseline rule
- [x] Added score, reason code and action label
- [x] Saved baseline_action_score.csv
- [x] Reviewed top 10 results
- [x] No future or label-derived features used